# Set up a regional MOM6 run within CESM framework

There are three main sections: 
1. First we create the MOM6 experiment using the [regional-mom6](https://github.com/COSIMA/regional-mom6/) package. This puts everything we need into two repositories, referred to as the `mom_run_dir` and `mom_input_dir`. These contain all of the configuration files (`MOM_input` etc.) and netcdf input files (`hgrid.nc`, OBC segments etc.) respectively. An `experiment` object is also created within the notebook, containing all the information we need about the regional mom6 setup to pass onto CESM.

2. A new MOM6 CESM case is created in the usual way by cloning the CESM repo and running the `create_case` command. 

3. We modify the new CESM case to work with the regional mom6 configuration we prepared earlier. We pass the path to this CESM run and the experiment object to the `setup_cesm` function, which carries out all the required modifications 

This is very much a work in progress! The goal in the short term was to get something working and make it nice later

# SECTION 1: Setup up you MOM6 regional experiment

This follows the normal workflow, copied from the `reanalysis_forced.ipynb` demo of [regional-mom6](https://github.com/COSIMA/regional-mom6/) but modified for a domain around Hawaii. See the documentation of the package for details and documentation

In [1]:
import sys
sys.path.insert(0, './regional-mom6')
%load_ext autoreload
%autoreload 2
import regional_mom6 as rmom6

import os
from pathlib import Path

In [2]:
expt = rmom6.load_experiment("/glade/u/home/manishrv/documents/nwa12_0.1/regional_mom_workflows/rm6/hawaii-clean-demo-tides-v2/run_files/config.json")

Reading from config file....
Creating Empty Experiment Object....
Setting Default Variables.....
Checking for hgrid and vgrid....
Found
Found
Checking for bathymetry...
Found
Checking for ocean state files....
Found
Checking for initial condition files....
Found
Checking for tides files....
Found


# SECTION 2: Create a blank CESM run

So far I've used Alper's [GUI](https://github.com/ESMCI/visualCaseGen?tab=readme-ov-file) branch of CESM. Clone respective branch of CESM and then run the generate case command. Below is the command I used to generate a global, MOM6 only run forced with JRA data atmosphere

`/glade/u/home/abarnes/cesm-runs/visualCaseGen/cesm2_3_beta17_gui/cime/scripts/create_newcase --compset 1850_DATM%JRA_SLND_SICE_MOM6_SROF_SGLC_SWAV --res TL319_t232 --case /glade/u/home/manishrv/cases/hawaii_clean_demo_tides_v3 --machine derecho --run-unsupported --project p93300612 --non-local`

In [7]:
# Save the path where the new CESM config lives
CESM_path = Path(f"/glade/u/home/manishrv/cases/hawaii_clean_demo_tides_v2")

# SECTION 3: Modify the CESM run to make it regional

In [8]:
from regional_casegen.cesm_tools import setup_cesm

setup_cesm(expt, CESM_path, project = "p93300612")

Copying input.nml, diag_table, MOM_input_and MOM_override to /glade/u/home/manishrv/cases/hawaii_clean_demo_tides_v2/SourceMods/src.mom
Adding NIGLOBAL = 340, NJGLOBAL = 236, and INPUTDIR = /glade/u/home/manishrv/documents/nwa12_0.1/regional_mom_workflows/rm6/hawaii-clean-demo-tides-v2/inputs to MOM_override
Removing references to MOM_layout in input.nml
Add MOM_override to parameter_filename in input.nml
Move all of the forcing files out of the forcing directory to the main inputdir
Find and replace instances of forcing/ with nothing in the MOM_input file
Find and replace instances of forcing/ with nothing in the MOM_input file
Make ESMF grid and save to inputdir
Make xml changes. Setting OCN_NX=340, OCN_NY=236
MOM6_MEMORY_MODE=dynamic_symmetric
OCN_DOMAIN_MESH, ICE_DOMAIN_MESH, MASK_MESH, MASK_GRID, OCN_GRID, ICE_GRID =/glade/u/home/manishrv/documents/nwa12_0.1/regional_mom_workflows/rm6/hawaii-clean-demo-tides-v2/inputs/esmf_mesh.nc
RUN_REFDATE, RUN_STARTDATE = 2020-01-01
For your c

Now hopefully doing the usual 

`./case.setup && ./case.build && ./case.submit` 

should at least run. Of course from here you'll have a lot of other things to fiddle around with to make it run *well!* 